# Pengujian Google Colab — Game Tebak Kata Bahasa Indonesia

**Tujuan:** membuktikan release versi 3.1.0 dapat diekstrak, divalidasi, diuji, dan dijalankan pada Google Colab.

Gunakan runtime standar **CPU / Hardware accelerator: None**. Jalankan setiap cell secara berurutan dari atas ke bawah.

> Upload ZIP release bersih: `game_tebak_kata_bahasa_indonesia_manual_test_verified_v3.1.0.zip`.


## 1. Catat lingkungan Colab

Ambil screenshot hasil cell ini sebagai bukti versi Python dan sistem operasi Colab.


In [ ]:
import platform
import sys
from datetime import datetime, timezone

print('Tanggal pengujian (UTC):', datetime.now(timezone.utc).isoformat())
print('Python:', sys.version)
print('Platform:', platform.platform())


## 2. Upload ZIP release

Saat tombol pemilihan file muncul, pilih ZIP release bersih dari komputer.


In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload tepat satu file ZIP. Ditemukan: {zip_names}')

ZIP_PATH = Path('/content') / zip_names[0]
print('ZIP terunggah:', ZIP_PATH)
print('Ukuran:', ZIP_PATH.stat().st_size, 'byte')


## 3. Ekstrak dan cari folder proyek


In [ ]:
import shutil
from pathlib import Path

EXTRACT_ROOT = Path('/content/tebak_kata_colab_test')
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_ROOT))
main_files = list(EXTRACT_ROOT.rglob('main.py'))
if len(main_files) != 1:
    raise RuntimeError(f'Harus ditemukan tepat satu main.py. Ditemukan: {main_files}')

PROJECT_DIR = main_files[0].parent
print('Folder proyek:', PROJECT_DIR)
print('Isi utama:')
for item in sorted(PROJECT_DIR.iterdir()):
    print('-', item.name)


## 4. Preflight release bersih

Cell ini memastikan leaderboard distribusi kosong dan tidak ada artefak runtime di dalam ZIP.


In [ ]:
import json

leaderboard_path = PROJECT_DIR / 'data' / 'leaderboard.json'
leaderboard = json.loads(leaderboard_path.read_text(encoding='utf-8'))
assert leaderboard == [], f'Leaderboard distribusi tidak kosong: {leaderboard}'

forbidden = []
for path in PROJECT_DIR.rglob('*'):
    if path.name == '__pycache__' or path.suffix == '.pyc' or path.name in {'.coverage', 'application.log'}:
        forbidden.append(str(path.relative_to(PROJECT_DIR)))

assert not forbidden, f'Artefak runtime ditemukan: {forbidden}'
print('PASS: leaderboard kosong')
print('PASS: tidak ada __pycache__, .pyc, .coverage, atau application.log')


## 5. Cek versi aplikasi


In [ ]:
import subprocess
import sys

version_result = subprocess.run(
    [sys.executable, 'main.py', '--version'],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)
print(version_result.stdout, end='')
if version_result.stderr:
    print(version_result.stderr, end='')
assert version_result.returncode == 0
assert '3.1.0' in version_result.stdout


## 6. Jalankan 52 automated test

Ambil screenshot bagian akhir yang menampilkan `Ran 52 tests` dan `OK`.


In [ ]:
test_result = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)
print(test_result.stdout, end='')
print(test_result.stderr, end='')
assert test_result.returncode == 0, 'Automated test gagal.'


## 7. Jalankan validator bank data

Ambil screenshot hasil validator.


In [ ]:
validator_result = subprocess.run(
    [sys.executable, 'tools/validate_bank.py'],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True,
)
print(validator_result.stdout, end='')
print(validator_result.stderr, end='')
assert validator_result.returncode == 0, 'Validator bank data gagal.'


## 8. Smoke test interaktif

Jalankan cell ini dan lakukan alur singkat:

1. masukkan nama `Ariq`;
2. buka `Cara bermain`, lalu kembali;
3. mulai satu ronde;
4. masukkan satu huruf;
5. ketik `0` untuk menyerah;
6. kembali ke menu;
7. pilih `0` untuk keluar.

Gunakan `%run`, bukan `!python`, agar `input()` dapat digunakan secara interaktif di notebook.


In [ ]:
import os

os.chdir(PROJECT_DIR)
%run main.py --no-clear


## 9. Buat hasil pengujian Colab

Jalankan setelah smoke test selesai. File hasil dapat diunduh sebagai bukti tambahan.


In [ ]:
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path

result_text = f'''HASIL PENGUJIAN GOOGLE COLAB
Tanggal UTC       : {datetime.now(timezone.utc).isoformat()}
Python            : {sys.version.split()[0]}
Platform          : {platform.platform()}
Versi aplikasi    : 3.1.0
Automated test    : LULUS
Validator data    : LULUS
Smoke test manual : ISI SETELAH DIPERIKSA — LULUS / GAGAL
Penguji           : Ariq
Catatan           : Isi bila ada perbedaan tampilan atau masalah.
'''

result_path = Path('/content/HASIL_TEST_GOOGLE_COLAB.txt')
result_path.write_text(result_text, encoding='utf-8')
print(result_text)
print('File hasil:', result_path)


## 10. Unduh hasil

Sebelum mengunduh, edit teks `Smoke test manual` pada cell sebelumnya bila perlu, lalu jalankan ulang cell tersebut.


In [ ]:
from google.colab import files
files.download('/content/HASIL_TEST_GOOGLE_COLAB.txt')
